In [1]:
!pip install -q datasets accelerate

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [3]:
from datasets import load_dataset
import pandas as pd, html, re
from sklearn.metrics import classification_report, confusion_matrix, f1_score

sent = load_dataset("cardiffnlp/tweet_eval", "sentiment")
test = sent["test"].to_pandas()
names = ["negative", "neutral", "positive"]

def fix_text(t):
    t = re.sub(r"\\u([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), t)
    return html.unescape(t)

def report(y_true, y_pred, title):
    print(title)
    print("Macro-F1:", round(f1_score(y_true, y_pred, average="macro"), 3))
    print(classification_report(y_true, y_pred, target_names=names, digits=3))
    print(pd.DataFrame(confusion_matrix(y_true, y_pred),
                       index=[f"true_{n}" for n in names],
                       columns=[f"pred_{n}" for n in names]))

test["text_fixed"] = test["text"].apply(fix_text)

README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.78MB            

sentiment/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  901kB            

sentiment/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/validation-00000-of-00001.parq(…): reconstructing file:   0%|          |  0.00B /  167kB            

sentiment/validation-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [4]:
from transformers import pipeline

clf = pipeline("sentiment-analysis",
               model="cardiffnlp/twitter-roberta-base-sentiment-latest",
               device=0, batch_size=64, truncation=True, max_length=128)

out = clf(test["text_fixed"].tolist())
label_map = {"negative": 0, "neutral": 1, "positive": 2}
test["rob_pred"] = [label_map[o["label"].lower()] for o in out]

report(test["label"], test["rob_pred"], "Twitter-RoBERTa (ready-made) on TweetEval test")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Twitter-RoBERTa (ready-made) on TweetEval test
Macro-F1: 0.724
              precision    recall  f1-score   support

    negative      0.689     0.807     0.743      3972
     neutral      0.757     0.658     0.704      5937
    positive      0.710     0.741     0.725      2375

    accuracy                          0.722     12284
   macro avg      0.719     0.735     0.724     12284
weighted avg      0.726     0.722     0.721     12284

               pred_negative  pred_neutral  pred_positive
true_negative           3204           711             57
true_neutral            1372          3904            661
true_positive             71           545           1759


In [5]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import numpy as np

MODEL = "distilroberta-base"
tok = AutoTokenizer.from_pretrained(MODEL)

def prep(batch):
    texts = [fix_text(t) for t in batch["text"]]
    return tok(texts, truncation=True, max_length=128)

ds = sent.map(prep, batched=True)
ds = ds.remove_columns(["text"]).rename_column("label", "labels")

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/45615 [00:00<?, ? examples/s]

Map:   0%|          | 0/12284 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2})

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="out", num_train_epochs=2,
    per_device_train_batch_size=32, per_device_eval_batch_size=64,
    learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="no",
    fp16=True, report_to="none", logging_steps=200)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds["train"], eval_dataset=ds["validation"],
                  processing_class=tok,
                  data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=compute_metrics)

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.613450,0.605200,0.713511
2,0.533043,0.610614,0.709999


TrainOutput(global_step=2852, training_loss=0.6025870806037293, metrics={'train_runtime': 147.4628, 'train_samples_per_second': 618.665, 'train_steps_per_second': 19.34, 'total_flos': 1036789849724454.0, 'train_loss': 0.6025870806037293, 'epoch': 2.0})

In [8]:
pred = trainer.predict(ds["test"])
test["ft_pred"] = np.argmax(pred.predictions, axis=1)
report(test["label"], test["ft_pred"], "Fine-tuned DistilRoBERTa on TweetEval test")

from google.colab import drive
drive.mount('/content/drive')
SAVE = "/content/drive/MyDrive/My_Projects/EmotionLens/distilroberta_tweeteval"
trainer.save_model(SAVE)
tok.save_pretrained(SAVE)
print("saved to", SAVE)

Fine-tuned DistilRoBERTa on TweetEval test
Macro-F1: 0.702
              precision    recall  f1-score   support

    negative      0.693     0.768     0.729      3972
     neutral      0.746     0.630     0.683      5937
    positive      0.634     0.767     0.694      2375

    accuracy                          0.701     12284
   macro avg      0.691     0.721     0.702     12284
weighted avg      0.707     0.701     0.700     12284

               pred_negative  pred_neutral  pred_positive
true_negative           3050           790            132
true_neutral            1279          3739            919
true_positive             70           484           1821
Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/drive/MyDrive/My_Projects/EmotionLens/distilroberta_tweeteval


In [9]:
OUT = "/content/drive/MyDrive/My_Projects/EmotionLens/test_predictions.csv"
test[["text", "text_fixed", "label", "rob_pred", "ft_pred"]].to_csv(OUT, index=False)
print("saved", len(test), "rows")

saved 12284 rows


In [10]:
import gc, torch

def run(lr, wd, epochs=3, seed=42):
    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=3,
        id2label={0: "negative", 1: "neutral", 2: "positive"},
        label2id={"negative": 0, "neutral": 1, "positive": 2})
    a = TrainingArguments(
        output_dir="tmp", num_train_epochs=epochs,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=lr, weight_decay=wd, eval_strategy="epoch",
        save_strategy="no", fp16=True, report_to="none",
        seed=seed, logging_strategy="no")
    t = Trainer(model=m, args=a, train_dataset=ds["train"],
                eval_dataset=ds["validation"], processing_class=tok,
                data_collator=DataCollatorWithPadding(tok),
                compute_metrics=compute_metrics)
    t.train()
    scores = [round(x["eval_macro_f1"], 4) for x in t.state.log_history if "eval_macro_f1" in x]
    del t, m
    gc.collect(); torch.cuda.empty_cache()
    return scores

results = {}
for lr, wd in [(2e-5, 0.01), (3e-5, 0.01), (5e-5, 0.01), (2e-5, 0.1)]:
    results[(lr, wd)] = run(lr, wd)
    print("lr", lr, "weight_decay", wd, "-> validation macro-F1 per epoch:", results[(lr, wd)])

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.611453,0.708769
2,No log,0.614681,0.712780
3,No log,0.637599,0.707929


lr 2e-05 weight_decay 0.01 -> validation macro-F1 per epoch: [0.7088, 0.7128, 0.7079]


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

...

lr 5e-05 weight_decay 0.01 -> validation macro-F1 per epoch: [0.6932, 0.71, 0.709]


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.619288,0.706797
2,No log,0.620159,0.708285
3,No log,0.643111,0.716585


lr 2e-05 weight_decay 0.1 -> validation macro-F1 per epoch: [0.7068, 0.7083, 0.7166]
